# AdaSigBoost — GPU-Accelerated Google Colab Runner (50-Trial Optuna + Family E Baselines)

This notebook runs the full **50-trial Optuna hyperparameter sweep** across **all 14 models** (`AdaSigBoost`, raw GBDTs, static decomposition hybrids, random kernels, learnable neural filterbanks, `PatchTST`, `ModernTCN`, `TimesNet`, `iTransformer`) on a Colab T4/A100 GPU, regenerates publication tables/figures, and exports `results_gpu.zip`.

In [ ]:
#@title 1. Check GPU & Clone AdaSigBoost Repository
!nvidia-smi || echo 'No GPU attached — select Runtime -> Change runtime type -> T4 GPU'
!rm -rf AdaSigBoost && git clone https://github.com/Vishal442005/AdaSigBoost.git
%cd AdaSigBoost

In [ ]:
#@title 2. Compile C++17/OpenMP Extension & Install Benchmark Dependencies
!pip install -q scikit-build-core pybind11 cmake ninja PyYAML optuna catboost xgboost lightgbm statsmodels PyWavelets vmdpy EMD-signal
!pip install -q --no-build-isolation -e .
!PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 python -m pytest -q

In [ ]:
#@title 3. Run GPU-Accelerated 14-Baseline x 50-Trial Optuna Benchmark
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

!python -m benchmarks.run_benchmark \
  --folds 5 \
  --trials 50 \
  --max-samples 3000 \
  --models AdaSigBoost XGBoost LightGBM CatBoost DWT+LightGBM VMD+CatBoost MultiRocket+Ridge MultiRocket+LightGBM LearnableFilter+MLP LearnableFilter+LightGBM PatchTST ModernTCN TimesNet iTransformer

In [ ]:
#@title 4. Run Synthetic Suite, Ablations, Case Studies, OOD Stress Tests & Regenerate Paper Tables
!python -m benchmarks.run_synthetic_suite
!python -m benchmarks.run_ablations
!python -m benchmarks.case_studies
!python -m benchmarks.ood_stress
!python -m paper.scripts.generate_tables
!zip -r results_gpu.zip results/ paper/tables/ paper/figures/
from google.colab import files
files.download('results_gpu.zip')